# 01 — Registration basics on one patient (HCC-TACE-Seg)

**Goal:** take one patient's **before-TACE** and **after-TACE** CT scans, line them up (registration), and see how well it worked.

We do it in 3 levels, from simplest to most flexible:

| Level | What it can do | Analogy |
|---|---|---|
| **No registration** | Nothing, just overlay the scans | Two photos stacked as they are |
| **Rigid** | Shift + rotate the whole scan | Sliding and turning a photo on a table |
| **Deformable (B-spline)** | Also bend/stretch locally | Stretching a rubber sheet so shapes match |

After each level we compute a **difference image** (after − before). If alignment is good, healthy tissue cancels to ≈ 0 (grey) and only real change remains.

**Runs on:** CPU only, ~15 GB RAM is plenty. Needs about 250 MB of disk for one patient.

## Step 0 — Settings

Change `PATIENT_DIR` if you put the data somewhere else. The folder must look like the TCIA download:

```
HCC_002/
 ├── 09-26-1997-NA-ABDPELVIS-84861/      ← session 1 (before TACE)
 │    ├── 2.000000-PRE LIVER-09935/      ← one series = one folder of .dcm slices
 │    ├── 4.000000-Recon 2 LIVER 3 PHASE AP-85789/
 │    └── 300.000000-Segmentation-74386/ ← expert masks (DICOM-SEG)
 └── 03-05-1998-NA-ABDPEL  LIVER-93301/  ← session 2 (after TACE)
```

In [ ]:
import os
from pathlib import Path

PATIENT_DIR = Path.home() / "Vansh_HA26001" / "data" / "HCC_002"
OUT_DIR = Path.home() / "Vansh_HA26001" / "results" / "01_registration_basics" / PATIENT_DIR.name
SERIES_KEYWORD = "3 PHASE"   # pick the series whose description contains this text

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data   :", PATIENT_DIR, "exists =", PATIENT_DIR.exists())
print("results:", OUT_DIR)

In [ ]:
import time
from collections import defaultdict
from datetime import datetime

import numpy as np
import pydicom
import SimpleITK as sitk
import matplotlib.pyplot as plt

# a shared server has many CPUs but you only get a share; 8 threads is a safe choice
sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(8)

WIN_LO, WIN_HI = -160, 240   # abdomen "soft-tissue window": HU range we display
print("SimpleITK", sitk.Version_VersionString(), "| pydicom", pydicom.__version__)

## Step 1 — What is in the patient folder?

**DICOM basics:** a CT scan is saved as many `.dcm` files, **one file per 2D slice**. A *series* is one folder of slices = one 3D scan. A *study/session* is one hospital visit and can contain several series (different contrast phases or reconstructions).

Pixel values are in **Hounsfield Units (HU)**: air ≈ −1000, fat ≈ −100, water = 0, liver ≈ 50–100 (higher with contrast), bone > 300.

In [ ]:
def session_date(d):
    return datetime.strptime(d.name[:10], "%m-%d-%Y")

sessions = sorted([d for d in PATIENT_DIR.iterdir() if d.is_dir()], key=session_date)
for s in sessions:
    print(f"\n=== session {session_date(s):%Y-%m-%d}  ({s.name})")
    for series in sorted(s.iterdir()):
        files = list(series.glob("*.dcm"))
        h = pydicom.dcmread(files[0], stop_before_pixels=True)
        print(f"   {h.Modality:4s} {len(files):4d} files   {series.name}")

gap = (session_date(sessions[-1]) - session_date(sessions[0])).days
print(f"\nTime between first and last session: {gap} days")

## Step 2 — Load the CT volumes (and a trap!)

**Trap found in this dataset:** a series called *"LIVER 3 PHASE"* can contain **two contrast phases stacked in one folder**: the same slice positions appear twice (e.g. arterial then portal-venous). If you load the folder naively, the two phases get mixed into one broken volume.

**Fix:** group the slices by the DICOM tag `AcquisitionNumber`, so each group = one phase = one clean 3D volume.

In [ ]:
def slice_position(h):
    # distance of the slice along the scan axis (normal of the slice plane)
    iop = np.array(h.ImageOrientationPatient, float)
    normal = np.cross(iop[:3], iop[3:])
    return float(np.dot(normal, np.array(h.ImagePositionPatient, float)))

def load_acquisitions(series_dir):
    # returns {AcquisitionNumber: sitk.Image}, one clean 3D volume per phase
    groups = defaultdict(list)
    for f in series_dir.glob("*.dcm"):
        h = pydicom.dcmread(f, stop_before_pixels=True)
        groups[int(getattr(h, "AcquisitionNumber", 0) or 0)].append((slice_position(h), str(f)))
    vols = {}
    for acq, items in sorted(groups.items()):
        items.sort()
        positions = [p for p, _ in items]
        if len(set(np.round(positions, 2))) != len(positions):
            print(f"   acquisition {acq}: duplicate slice positions, skipped")
            continue
        reader = sitk.ImageSeriesReader()
        reader.SetFileNames([f for _, f in items])
        vols[acq] = sitk.Cast(reader.Execute(), sitk.sitkFloat32)
    return vols

def describe(name, img):
    sx, sy, sz = img.GetSize()
    px = img.GetSpacing()
    print(f"{name}: {sx}x{sy}x{sz} voxels | spacing {px[0]:.2f} x {px[1]:.2f} x {px[2]:.2f} mm "
          f"| covers {sz * px[2]:.0f} mm along the body")

def find_series(session, keyword, modality="CT"):
    for series in sorted(session.iterdir()):
        h = pydicom.dcmread(next(series.glob("*.dcm")), stop_before_pixels=True)
        if h.Modality == modality and keyword.upper() in str(getattr(h, "SeriesDescription", "")).upper():
            return series
    raise FileNotFoundError(f"no {modality} series with '{keyword}' in {session.name}")

base_series = find_series(sessions[0], SERIES_KEYWORD)
follow_series = find_series(sessions[-1], SERIES_KEYWORD)
base_vols = load_acquisitions(base_series)
follow_vols = load_acquisitions(follow_series)

print("BEFORE (baseline):", base_series.name)
for acq, v in base_vols.items(): describe(f"   acquisition {acq}", v)
print("AFTER (follow-up):", follow_series.name)
for acq, v in follow_vols.items(): describe(f"   acquisition {acq}", v)

### Choose which phase to compare

Each session has 2 acquisitions. The **first acquisition is usually the earlier contrast phase** (arterial) and the later one the portal-venous phase. That is an educated guess: this dataset does not store the phase name per acquisition, so **look at the pictures below to check.**

Rule: **compare the same phase with the same phase.** Otherwise contrast differences look like "changes".

In [ ]:
def to_display(img):
    return np.clip(sitk.GetArrayFromImage(img), WIN_LO, WIN_HI)   # array order is (z, y, x)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for row, (label, vols) in enumerate([("BEFORE", base_vols), ("AFTER", follow_vols)]):
    for col, (acq, v) in enumerate(list(vols.items())[:2]):
        a = to_display(v)
        axes[row, col].imshow(a[a.shape[0] // 2], cmap="gray")
        axes[row, col].set_title(f"{label} - acquisition {acq} (middle slice)")
        axes[row, col].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
BASE_ACQ = list(base_vols)[0]      # change these if the pictures show a different match
FOLLOW_ACQ = list(follow_vols)[0]

fixed = base_vols[BASE_ACQ]        # "fixed"  = the reference we align TO   (before)
moving = follow_vols[FOLLOW_ACQ]   # "moving" = the scan we move/bend        (after)
describe("fixed  (before)", fixed)
describe("moving (after) ", moving)

## Step 3 — Load the expert masks (DICOM-SEG)

A radiologist outlined 4 structures on the **before** scan: Liver, Mass (= tumour), Portal vein, Abdominal aorta. DICOM-SEG stores them as frames with their 3D positions; we rebuild a 3D label volume and put it on the same grid as `fixed`.

(Function adapted from `processing/HCC-TACE-Seg/pipeline/02_seg_to_mask.py`.)

In [ ]:
def load_seg(seg_path):
    ds = pydicom.dcmread(seg_path)
    names = {int(s.SegmentNumber): str(s.SegmentLabel) for s in ds.SegmentSequence}
    frames = ds.pixel_array
    if frames.ndim == 2:
        frames = frames[None]
    sfg = ds.SharedFunctionalGroupsSequence[0]
    ps = [float(x) for x in sfg.PixelMeasuresSequence[0].PixelSpacing]
    iop = [float(x) for x in sfg.PlaneOrientationSequence[0].ImageOrientationPatient]
    row_dir, col_dir = np.array(iop[:3]), np.array(iop[3:])
    normal = np.cross(row_dir, col_dir)
    rec = []
    for i, pf in enumerate(ds.PerFrameFunctionalGroupsSequence):
        seg_n = int(pf.SegmentIdentificationSequence[0].ReferencedSegmentNumber)
        pos = np.array([float(x) for x in pf.PlanePositionSequence[0].ImagePositionPatient])
        rec.append((seg_n, pos, i))
    # round once and look slices up in that same array (mixing two rounding methods can crash)
    proj = np.round([np.dot(p, normal) for _, p, _ in rec], 2)
    uniq = np.unique(proj)
    slice_of = np.searchsorted(uniq, proj)
    dz = float(np.median(np.diff(uniq))) if len(uniq) > 1 else 1.0
    vol = np.zeros((len(uniq), frames.shape[1], frames.shape[2]), np.uint8)
    # stamp the tumour ("Mass") last so it wins where structures overlap
    for seg_n, pos, i in sorted(rec, key=lambda r: names.get(r[0], "").lower() == "mass"):
        vol[slice_of[i]][frames[i] > 0] = seg_n
    img = sitk.GetImageFromArray(vol)
    img.SetSpacing((ps[1], ps[0], abs(dz)))
    img.SetDirection([float(x) for x in np.column_stack([row_dir, col_dir, normal]).flatten()])
    img.SetOrigin([float(x) for x in rec[int(np.argmin(proj))][1]])
    return img, names

seg_series = find_series(sessions[0], "", modality="SEG")
seg_img, seg_names = load_seg(next(seg_series.glob("*.dcm")))
# put the labels on the fixed grid (nearest neighbour keeps labels as whole numbers)
labels = sitk.GetArrayFromImage(sitk.Resample(seg_img, fixed, sitk.Transform(), sitk.sitkNearestNeighbor, 0))
name_to_id = {v.lower(): k for k, v in seg_names.items()}
liver = labels == name_to_id["liver"]
tumour = labels == name_to_id["mass"]

vox_ml = np.prod(fixed.GetSpacing()) / 1000
print("segments:", seg_names)
print(f"liver volume  : {liver.sum() * vox_ml:7.1f} ml")
print(f"tumour volume : {tumour.sum() * vox_ml:7.1f} ml")

# slice to show in all figures = the slice with the most tumour
SHOW_Z = int(np.argmax(tumour.sum(axis=(1, 2)))) if tumour.any() else fixed.GetSize()[2] // 2
print("figures will show slice", SHOW_Z)

## Step 4 — Helpers: resampling, difference image, and scores

**Resampling** = "re-draw the moving scan on the fixed scan's grid", using a transform (the recipe for where each point moves). With the *identity* transform, nothing moves, it just uses the scanner's own coordinates.

**Scores** (computed inside the **liver**, with the tumour cut out, because we want *healthy* tissue to match):

| Score | Meaning | Better |
|---|---|---|
| **MAE (HU)** | average absolute brightness difference | lower |
| **NCC** | how similar the patterns are (−1…1) | higher |
| **PSNR (dB)** | image similarity, the same measure the team pipeline reports | higher |

In [ ]:
from scipy import ndimage

fixed_arr = sitk.GetArrayFromImage(fixed)
# healthy-liver region: liver minus a grown tumour (grow by 3 voxels as a safety margin)
healthy_liver = liver & ~ndimage.binary_dilation(tumour, iterations=3)

def resample(moving_img, transform):
    return sitk.Resample(moving_img, fixed, transform, sitk.sitkLinear, -1024.0, sitk.sitkFloat32)

def scores(warped_arr, region):
    # only voxels that the warped scan actually covers (outside its field of view it is -1024)
    m = region & (warped_arr > -1000)
    a = np.clip(fixed_arr[m], WIN_LO, WIN_HI)
    b = np.clip(warped_arr[m], WIN_LO, WIN_HI)
    mse = np.mean((a - b) ** 2)
    ncc = np.corrcoef(a, b)[0, 1]
    return {"MAE_HU": float(np.mean(np.abs(a - b))), "NCC": float(ncc),
            "PSNR_dB": float(20 * np.log10(WIN_HI - WIN_LO) - 10 * np.log10(mse)),
            "coverage_%": float(100 * m.sum() / region.sum())}

def show(warped_arr, title, fname):
    f = np.clip(fixed_arr[SHOW_Z], WIN_LO, WIN_HI)
    w = np.clip(warped_arr[SHOW_Z], WIN_LO, WIN_HI)
    fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))
    ax[0].imshow(f, cmap="gray"); ax[0].set_title("BEFORE (fixed)")
    ax[1].imshow(w, cmap="gray"); ax[1].set_title("AFTER (moved onto BEFORE)")
    # checkerboard: alternating squares from each image; edges should continue smoothly
    tiles = (np.indices(f.shape) // 64).sum(axis=0) % 2 == 0
    ax[2].imshow(np.where(tiles, f, w), cmap="gray"); ax[2].set_title("checkerboard (look at the edges)")
    ax[3].imshow(w - f, cmap="bwr", vmin=-200, vmax=200); ax[3].set_title("difference AFTER - BEFORE (HU)")
    for a in ax:
        a.contour(tumour[SHOW_Z], colors="yellow", linewidths=1)
        a.contour(liver[SHOW_Z], colors="lime", linewidths=0.6)
        a.axis("off")
    fig.suptitle(title + "   (lime = liver, yellow = tumour, both drawn on BEFORE)")
    plt.tight_layout(); plt.savefig(OUT_DIR / fname, dpi=90); plt.show()

results = {}

## Step 5 — Level 0: no registration

We just draw the AFTER scan in the BEFORE scan's coordinates. The patient lay differently on the table, so expect a **messy difference image** with strong red/blue edges everywhere.

In [ ]:
identity = sitk.Transform(3, sitk.sitkIdentity)
warped_none = sitk.GetArrayFromImage(resample(moving, identity))
results["none"] = scores(warped_none, healthy_liver)
print(results["none"])
show(warped_none, "Level 0 - no registration", "level0_none.png")

## Step 6 — Level 1: rigid registration

**Rigid** = 6 numbers: 3 shifts (x, y, z) + 3 rotations. The computer tries many values and keeps the one where the images match best.

How it works:
1. **Start guess:** align the centres of the two bodies (so we start close).
2. **Similarity metric:** *Mutual Information*, which measures "when BEFORE shows liver, does AFTER also show liver?". It works even if brightness differs a bit (contrast timing).
3. **Optimizer:** repeatedly nudges the 6 numbers in the direction that improves the metric.
4. **Coarse → fine (pyramid):** first on a blurry, 4× smaller version (fast, finds the big shift), then 2×, then full size (fine details).

In [ ]:
def body_mask(img):
    return sitk.Cast(sitk.BinaryThreshold(img, -500, 3000), sitk.sitkUInt8)   # everything denser than air

def clamp(img):
    return sitk.Clamp(img, sitk.sitkFloat32, -1000, 1500)   # ignore extreme values (metal, air noise)

curve = []
def run_registration(reg, f_img, m_img):
    curve.clear()
    reg.AddCommand(sitk.sitkIterationEvent, lambda: curve.append(reg.GetMetricValue()))
    t0 = time.time()
    tx = reg.Execute(f_img, m_img)
    print(f"done in {time.time() - t0:.0f} s, {len(curve)} iterations in total")
    return tx

# 1. start guess: match the centres of mass of the two body masks
init = sitk.CenteredTransformInitializer(
    sitk.Cast(body_mask(fixed), sitk.sitkFloat32), sitk.Cast(body_mask(moving), sitk.sitkFloat32),
    sitk.Euler3DTransform(), sitk.CenteredTransformInitializerFilter.MOMENTS)

reg = sitk.ImageRegistrationMethod()
reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
reg.SetMetricSamplingStrategy(reg.RANDOM)
reg.SetMetricSamplingPercentage(0.02, seed=42)       # use 2 % of voxels: fast and good enough
reg.SetMetricFixedMask(body_mask(fixed))
reg.SetInterpolator(sitk.sitkLinear)
reg.SetOptimizerAsRegularStepGradientDescent(learningRate=2.0, minStep=1e-3, numberOfIterations=300)
reg.SetOptimizerScalesFromPhysicalShift()            # makes 1 mm shift and ~1 mm rotation comparable
reg.SetShrinkFactorsPerLevel([4, 2, 1])
reg.SetSmoothingSigmasPerLevel([2, 1, 0])
reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
reg.SetInitialTransform(init, inPlace=False)

result = run_registration(reg, clamp(fixed), clamp(moving))
rigid_curve = list(curve)

# the result comes wrapped in a container; take out the plain rigid transform
rigid = sitk.Euler3DTransform(result.GetNthTransform(0) if result.GetName() == "CompositeTransform" else result)
print("rotation (degrees):", np.round(np.degrees([rigid.GetAngleX(), rigid.GetAngleY(), rigid.GetAngleZ()]), 2))
print("shift (mm)        :", np.round(rigid.GetTranslation(), 1))

In [ ]:
warped_rigid = sitk.GetArrayFromImage(resample(moving, rigid))
results["rigid"] = scores(warped_rigid, healthy_liver)
print(results["rigid"])
show(warped_rigid, "Level 1 - rigid", "level1_rigid.png")

## Step 7 — Level 2: deformable (B-spline) registration

The liver is soft: breathing and stomach filling bend it. Rigid cannot fix bending.

**B-spline** puts a coarse **grid of control points** (like pins in a rubber sheet) over the scan. Each pin can move a little; the image stretches smoothly between pins. We start from the rigid result and only learn the extra bending.

⚠️ **The danger for HOPE:** a flexible warp can also "squash away" a real change, such as a small new tumour, because the computer sees it as a mismatch to remove. Coarse grids (few pins) bend less and are safer. We will test this properly in a later experiment (synthetic small lesions).

In [ ]:
GRID_NODES = [6, 6, 6]   # pins per direction: more = more flexible (and riskier)

bspline = sitk.BSplineTransformInitializer(fixed, GRID_NODES)
reg = sitk.ImageRegistrationMethod()
reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
reg.SetMetricSamplingStrategy(reg.RANDOM)
reg.SetMetricSamplingPercentage(0.02, seed=42)
reg.SetMetricFixedMask(body_mask(fixed))
reg.SetInterpolator(sitk.sitkLinear)
reg.SetOptimizerAsLBFGSB(gradientConvergenceTolerance=1e-5, numberOfIterations=100)
reg.SetShrinkFactorsPerLevel([4, 2])
reg.SetSmoothingSigmasPerLevel([2, 1])
reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
reg.SetMovingInitialTransform(rigid)            # start from the rigid answer
reg.SetInitialTransform(bspline, inPlace=True)  # only the bending is optimised

run_registration(reg, clamp(fixed), clamp(moving))   # optimises `bspline` in place
bspline_curve = list(curve)
deformable = sitk.CompositeTransform([rigid, bspline])   # rigid + bending together

In [ ]:
warped_def = sitk.GetArrayFromImage(resample(moving, deformable))
results["deformable"] = scores(warped_def, healthy_liver)
print(results["deformable"])
show(warped_def, f"Level 2 - deformable B-spline {GRID_NODES}", "level2_deformable.png")

## Step 8 — Compare the three levels

If registration helps: **MAE goes down, NCC and PSNR go up** from none → rigid → deformable.

`coverage_%` = how much of the liver the AFTER scan covers once aligned (the two scans do not always cover exactly the same body range).

In [ ]:
import csv

print(f"{'level':12s} {'MAE_HU':>8s} {'NCC':>7s} {'PSNR_dB':>8s} {'coverage_%':>11s}")
for level, r in results.items():
    print(f"{level:12s} {r['MAE_HU']:8.1f} {r['NCC']:7.3f} {r['PSNR_dB']:8.2f} {r['coverage_%']:11.1f}")

with open(OUT_DIR / "metrics.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["patient", "level", "MAE_HU", "NCC", "PSNR_dB", "coverage_pct"])
    for level, r in results.items():
        w.writerow([PATIENT_DIR.name, level, r["MAE_HU"], r["NCC"], r["PSNR_dB"], r["coverage_%"]])

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(rigid_curve); ax[0].set_title("rigid: metric per iteration (lower = better match)")
ax[1].plot(bspline_curve); ax[1].set_title("deformable: metric per iteration")
for a in ax: a.set_xlabel("iteration")
plt.tight_layout(); plt.savefig(OUT_DIR / "convergence.png", dpi=90); plt.show()

## Step 9 — First look at treatment response

Now that AFTER is aligned onto BEFORE, the **expert tumour outline from BEFORE also marks the tumour area on AFTER**. After TACE, the treated tumour often contains **lipiodol** (an oily contrast agent used in TACE), which looks very bright (high HU).

This is a first taste of experiment **E4 (treatment response from change maps)**.

In [ ]:
def tumour_stats(arr):
    v = arr[tumour & (arr > -1000)]
    return f"mean {v.mean():6.1f} HU | median {np.median(v):6.1f} HU | 95th percentile {np.percentile(v, 95):6.1f} HU"

print("tumour region BEFORE           :", tumour_stats(fixed_arr))
print("tumour region AFTER (deformable):", tumour_stats(warped_def))
print("healthy liver BEFORE           :", f"mean {fixed_arr[healthy_liver].mean():6.1f} HU")

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for a, arr, t in [(ax[0], fixed_arr, "BEFORE"), (ax[1], warped_def, "AFTER (aligned)")]:
    a.imshow(np.clip(arr[SHOW_Z], -160, 400), cmap="gray")
    a.contour(tumour[SHOW_Z], colors="yellow", linewidths=1)
    a.set_title(t + " - wider window to show bright lipiodol"); a.axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR / "tumour_before_after.png", dpi=90); plt.show()

In [ ]:
# save the transforms (tiny files) so results can be reproduced without re-running
sitk.WriteTransform(rigid, str(OUT_DIR / "rigid.tfm"))
sitk.WriteTransform(deformable, str(OUT_DIR / "deformable.tfm"))
print("saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## What to write down (your lab notebook)

1. The metrics table: did each level improve MAE / NCC / PSNR?
2. The difference images: where does red/blue remain after deformable? (edges of organs, bowel, tumour?)
3. The checkerboard: do the liver edges continue smoothly across squares?
4. The tumour: how did its HU change after TACE?

**Next:** turn this notebook into a script that runs over many patients (experiment **E1**), then the synthetic small-lesion test (**E5**).